# Recall of the retrieval

This notebook is for the check of week 5 and the check of week 6. It shows the recall in the top five results for each search mode and each language, and it helps you read the questions that a mode misses.

The notebook has no logic of its own. It calls `measure()` from `evals/recall.py` and `retrieve()` from `assistant/retrieval.py`, which you wrote. Postgres and the model services must run (`make up`).

Open it with `make notebook` from the repository root.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Find the repository root, so that `assistant` and `evals` can be imported.
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from assistant.retrieval import retrieve
from evals.recall import measure

lines = (ROOT / "evals" / "retrieval_questions.jsonl").read_text(encoding="utf-8").splitlines()
questions = [json.loads(line) for line in lines if line.strip()]
pd.Series([question["language"] for question in questions]).value_counts()

## Week 5: the vector search

`measure()` runs each question through the vector search. On a CPU this takes less than a minute.

In [ ]:
week5 = measure(questions, ["vector"], 5)
pd.DataFrame(week5["recall"]).T.round(2)

The questions that the vector search misses. Read some of them before week 6. For each one, find the cause: the expected chunk ID is wrong, the question uses other words than the document, or the chunk is one of many chunks that look the same.

In [ ]:
rows5 = pd.DataFrame(week5["rows"])
rows5[~rows5["found"]][["id", "language", "expected_chunk_ids", "retrieved_chunk_ids"]]

This function prints what one mode returns for one question. Use it to read a miss.

In [ ]:
def show(question_id: str, mode: str = "vector") -> None:
    question = next(question for question in questions if question["id"] == question_id)
    print(question["question"], "\nexpected:", question["expected_chunk_ids"], "\n")
    for result in retrieve(question["question"], mode, 5):
        mark = "*" if result["chunk_id"] in question["expected_chunk_ids"] else " "
        print(f"{mark} {result['score']:.3f}  {result['chunk_id']}\n    {result['text'][:200]}\n")


show(questions[0]["id"])

## Week 6: vector search, hybrid search, and hybrid search with re-ranking

The re-ranking model is slow on a CPU, so this part reads the file that `make recall` wrote. Run `make recall` first. It runs the 50 questions in the three modes and writes `evals/results/recall.json`.

In [ ]:
week6 = json.loads((ROOT / "evals" / "results" / "recall.json").read_text(encoding="utf-8"))
pd.DataFrame(week6["recall"]).T.round(2)

The questions where the modes do not agree. `True` means that an expected chunk is in the top five. Look for questions that the hybrid search finds and the vector search misses, and for questions that the re-ranking loses.

In [ ]:
found = pd.DataFrame(week6["rows"]).pivot(index="id", columns="mode", values="found")
languages = pd.Series({question["id"]: question["language"] for question in questions}, name="language")
found.join(languages)[found.nunique(axis=1) > 1]

Read one of these questions in each mode with `show("r01", "hybrid")` and `show("r01", "rerank")`. Then write the table above in `RESULTS.md` of this phase.